# 02 · NMF 분해 & 온도 의존성

**워크플로**
1. 01에서 저장한 온도별 `npz`(G(r))를 모두 불러오기
2. G(r)를 **무지개색 워터폴**로 쌓아 보기 (겹치지 않게 offset)
3. **NMF**(성분 2개, 수정 가능)로 분해 → 성분 1·2 프로파일, 성분별 온도 기여
4. **NMF 분율** vs 온도
5. **1st peak 위치 변화** — 1.5~1.7 Å 구간 가우시안 피팅

재사용 함수는 모두 `fourdstem`에 있습니다:
`decompose_profiles`, `plot_series_waterfall`, `plot_fractions`, `fit_gaussian_peak`.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

OUT_DIR = "rdf_npz"                 # 01 노트북의 저장 폴더
N_COMPONENTS = 2                    # NMF 성분 수 (먼저 2, 필요시 조정)
FIRST_PEAK_WINDOW = (1.5, 1.7)     # 1st peak 가우시안 피팅 구간 (Å)

## 1) NPZ 불러오기 (온도별 G(r))

온도 순으로 정렬하고, 공통 r 축으로 맞춰 행렬 `X (n_temperature × n_r)`를 만듭니다.

In [ ]:
files = sorted(glob.glob(os.path.join(OUT_DIR, "*_rdf.npz")))
assert files, f"{OUT_DIR}/ 에 npz가 없습니다. 먼저 01 노트북을 실행하세요."

records = []
for p in files:
    d = fds.load_result_npz(p)
    records.append((float(d["temperature"]), np.asarray(d["r"]), np.asarray(d["Gr"])))
records.sort(key=lambda t: t[0])

temps = np.array([t for t, _, _ in records])
r_ref = records[0][1]
# 공통 r축으로 보간(그리드가 다를 수 있으므로)
X = np.vstack([np.interp(r_ref, r, Gr) for _, r, Gr in records])
profiles = [(r_ref, X[i]) for i in range(len(temps))]
print("temperatures:", temps)
print("X shape (n_T × n_r):", X.shape)

## 2) 온도별 G(r) 무지개 워터폴

`plot_series_waterfall`은 온도에 따라 **무지개색**으로 칠하고, `offset`만큼 위로 쌓아 겹치지 않게 합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
fds.plot_series_waterfall(profiles, temps, ax=ax, cmap="rainbow",
                          xlabel="r (Å)", labels=[f"{int(T)}K" for T in temps])
ax.set_title("G(r) per temperature (rainbow waterfall)")
plt.show()

## 3) NMF 분해 (성분 2개)

`decompose_profiles(X, n_components=2)` → `X ≈ W · H`
- `components` (H): 성분 프로파일 (성분 1, 성분 2)
- `weights` (W), `fractions`: 온도별 성분 기여도 / 분율

In [ ]:
dec = fds.decompose_profiles(X, n_components=N_COMPONENTS, x=r_ref, method="nmf")
print("components:", dec.components.shape, " weights:", dec.weights.shape)

# 성분 1·2 프로파일
fig, ax = plt.subplots(figsize=(8, 4.5))
for j in range(N_COMPONENTS):
    ax.plot(r_ref, dec.components[j], lw=2, label=f"component {j+1}")
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel("r (Å)"); ax.set_ylabel("NMF component")
ax.set_title("NMF component profiles"); ax.legend()
plt.show()

### 3b) 성분별 온도 기여 워터폴 (무지개)

각 성분 `j`의 온도별 기여 `W[:, j] · H[j]`를 무지개 워터폴로 쌓습니다.
(성분 1은 저온에서, 성분 2는 고온에서 강해지는 것을 볼 수 있습니다.)

In [ ]:
fig, axes = plt.subplots(1, N_COMPONENTS, figsize=(6.5*N_COMPONENTS, 7), squeeze=False)
for j in range(N_COMPONENTS):
    contrib = [(r_ref, dec.weights[i, j]*dec.components[j]) for i in range(len(temps))]
    fds.plot_series_waterfall(contrib, temps, ax=axes[0][j], cmap="rainbow",
                              xlabel="r (Å)")
    axes[0][j].set_title(f"component {j+1} contribution vs T")
plt.tight_layout(); plt.show()

## 4) NMF 분율 vs 온도

각 온도에서 성분 분율(합=1)의 변화. 상전이/구조 변화 지점을 정량적으로 드러냅니다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
fds.plot_fractions(temps, dec.fractions, ax=ax, xlabel="temperature (K)",
                   comp_labels=[f"component {j+1}" for j in range(N_COMPONENTS)])
ax.set_title("NMF fraction vs temperature")
plt.show()

## 5) 1st peak 위치 변화 — 가우시안 피팅 (1.5~1.7 Å)

각 온도의 G(r)에서 첫 배위 피크를 **가우시안으로 피팅**해 중심을 추출하고, 온도에 따른 이동을 봅니다.
(SiOx의 Si–O 첫 배위 ≈ 1.6 Å)

In [ ]:
centers, sigmas = [], []
for (r, Gr) in profiles:
    fit = fds.fit_gaussian_peak(r, Gr, *FIRST_PEAK_WINDOW)
    centers.append(fit["center"]); sigmas.append(fit["sigma"])
centers = np.array(centers); sigmas = np.array(sigmas)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# (좌) 대표 온도들의 피팅 겹쳐 보기
cm = plt.get_cmap("rainbow")
for i in range(0, len(temps), max(1, len(temps)//5)):
    r, Gr = profiles[i]
    sel = (r >= FIRST_PEAK_WINDOW[0]-0.15) & (r <= FIRST_PEAK_WINDOW[1]+0.15)
    c = cm(i/max(len(temps)-1, 1))
    ax[0].plot(r[sel], Gr[sel], ".", color=c, ms=3)
    fit = fds.fit_gaussian_peak(r, Gr, *FIRST_PEAK_WINDOW)
    ax[0].plot(fit["xfit"], fit["yfit"], "-", color=c, lw=1.5,
               label=f"{int(temps[i])}K")
ax[0].axvspan(*FIRST_PEAK_WINDOW, color="0.9", zorder=0)
ax[0].set_xlabel("r (Å)"); ax[0].set_ylabel("G(r)")
ax[0].set_title("1st peak Gaussian fit"); ax[0].legend(fontsize=8)

# (우) 첫 피크 중심 vs 온도
ax[1].errorbar(temps, centers, yerr=sigmas, fmt="o-", color="crimson", capsize=3)
ax[1].set_xlabel("temperature (K)"); ax[1].set_ylabel("1st peak center (Å)")
ax[1].set_title("1st-neighbour distance vs T")
plt.tight_layout(); plt.show()

for T, c, s in zip(temps, centers, sigmas):
    print(f"  T={T:>5.0f}K   center={c:.3f} Å   σ={s:.3f} Å")

**정리** — 01에서 온도별 G(r)를 만들고, 02에서 NMF로 두 구조 성분을 분리해 분율의 온도 의존성을 보고,
첫 배위 피크의 이동을 가우시안으로 정량화했습니다. `N_COMPONENTS`, `FIRST_PEAK_WINDOW`만 바꿔 재사용하세요.